# Uczenie maszynowe - zadania 41-50: rozwiązania

__Kiedy wykonać:__ po przerobieniu notatnika `01_interpretowalnosc_modeli.ipynb` (rozdział 4.6), czyli po rozdziałach 4.1-4.6:

- zadania 41-42 - `00_drzewa_decyzyjne_wstęp.ipynb` (4.1),
- zadanie 43 - `01_niedouczenie_i_przeuczenie.ipynb` (4.3),
- zadanie 44 - `01_wprowadzenie.ipynb` (4.2),
- zadanie 45 - `02_dane_niezbalansowane.ipynb` (4.3),
- zadania 46-47 - `01_walidacja_krzyzowa.ipynb` (4.4),
- zadania 48-49 - `01_przeszukiwanie_siatki.ipynb` (4.5),
- zadanie 50 - `01_interpretowalnosc_modeli.ipynb` (4.6) oraz ważność cech z lasu z `01_wprowadzenie.ipynb` (4.2).

Część zadań ma kilka równoważnych rozwiązań - poniżej pokazano jedno z nich.

### Import bibliotek i dane

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import randint
from sklearn.datasets import load_breast_cancer, make_classification, make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV,
                                     StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

X, y = make_moons(n_samples=2000, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print(X_train.shape, X_test.shape)

(1500, 2) (500, 2)


### Ćwiczenie 41
Drzewo decyzyjne wybiera podziały, które najbardziej zmniejszają **nieczystość** węzła. Zaimplementuj dwie funkcje przyjmujące etykiety klas węzła:

$$\text{Gini} = 1 - \sum_k p_k^2 \qquad \text{Entropia} = -\sum_k p_k \log_2 p_k$$

gdzie $p_k$ to udział klasy $k$ w węźle. Wydrukuj Gini i entropię (3 miejsca) dla trzech węzłów z komórki poniżej.

__Oczekiwany wynik:__

```
czysty 0.0 0.0
mieszany 3:1 0.375 0.811
mieszany 2:2 0.5 1.0
```

__Wskazówka:__ Udziały klas policzysz przez `np.bincount(labels) / len(labels)`. Pomiń $p_k = 0$ (logarytm zera jest nieokreślony). Zapis $-\log_2 p$ jest równoważny $\log_2 (1/p)$.

In [2]:
nodes = {
    'czysty': np.array([0, 0, 0, 0]),
    'mieszany 3:1': np.array([0, 0, 0, 1]),
    'mieszany 2:2': np.array([0, 0, 1, 1]),
}

In [3]:
def gini(labels):
    p = np.bincount(labels) / len(labels)
    return 1 - (p ** 2).sum()


def entropy(labels):
    p = np.bincount(labels) / len(labels)
    p = p[p > 0]
    return (p * np.log2(1 / p)).sum()


for name, labels in nodes.items():
    print(name, round(gini(labels), 3), round(entropy(labels), 3))

czysty 0.0 0.0
mieszany 3:1 0.375 0.811
mieszany 2:2 0.5 1.0


### Ćwiczenie 42
Wytrenuj `DecisionTreeClassifier(random_state=42)` **bez ograniczeń** na danych `make_moons` (zbiory `X_train`, `X_test` z komórki z importami).

Wydrukuj: accuracy treningowe i testowe (3 miejsca), głębokość drzewa (`get_depth()`) i liczbę liści (`get_n_leaves()`). Jak nazywa się zjawisko, które widać w wynikach?

__Oczekiwany wynik:__

```
1.0 0.904
16 96
```

__Wskazówka:__ Drzewo bez ograniczeń „uczy się na pamięć” zbioru treningowego (przeuczenie).

In [4]:
tree = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
print(round(tree.score(X_train, y_train), 3), round(tree.score(X_test, y_test), 3))
print(tree.get_depth(), tree.get_n_leaves())

1.0 0.904
16 96


### Ćwiczenie 43
Narysuj (czyli wydrukuj) **krzywą złożoności**: dla `max_depth` od 1 do 10 wytrenuj drzewo (`random_state=42`) i wydrukuj accuracy treningowe oraz testowe (3 miejsca).

Na końcu wydrukuj głębokość, dla której accuracy **testowe** jest największe.

__Oczekiwany wynik:__

```
1 0.824 0.83
2 0.911 0.904
3 0.911 0.904
4 0.913 0.902
5 0.957 0.922
6 0.964 0.928
7 0.969 0.926
8 0.977 0.918
9 0.982 0.914
10 0.987 0.91
najlepsza głębokość: 6
```

__Wskazówka:__ Dla małej głębokości oba wyniki są niskie (niedouczenie), potem accuracy treningowe rośnie, a testowe przestaje rosnąć (przeuczenie).

In [5]:
best_depth, best_score = None, 0
for depth in range(1, 11):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    train_acc = tree.score(X_train, y_train)
    test_acc = tree.score(X_test, y_test)
    print(depth, round(train_acc, 3), round(test_acc, 3))
    if test_acc > best_score:
        best_depth, best_score = depth, test_acc
print('najlepsza głębokość:', best_depth)

1 0.824 0.83
2 0.911 0.904
3 0.911 0.904
4 0.913 0.902
5 0.957 0.922
6 0.964 0.928
7 0.969 0.926
8 0.977 0.918
9 0.982 0.914
10 0.987 0.91
najlepsza głębokość: 6


### Ćwiczenie 44
Porównaj pojedyncze drzewo z **lasem losowym**. Wytrenuj:
- `DecisionTreeClassifier(random_state=42)` (bez ograniczeń),
- `RandomForestClassifier(n_estimators=200, oob_score=True, random_state=42)`.

Wydrukuj accuracy testowe obu modeli (3 miejsca) oraz **ocenę OOB** lasu (`oob_score_`, 3 miejsca).

__Oczekiwany wynik:__

```
0.904 0.93
0.945
```

__Wskazówka:__ Ocena OOB korzysta z obserwacji, których dane drzewo nie zobaczyło podczas uczenia (próbkowanie ze zwracaniem).

In [6]:
tree = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
forest = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=42)
forest.fit(X_train, y_train)
print(round(tree.score(X_test, y_test), 3), round(forest.score(X_test, y_test), 3))
print(round(forest.oob_score_, 3))

0.904 0.93
0.945


### Ćwiczenie 45
Dane `X_imb`, `y_imb` są **mocno niezbalansowane** (ok. 5% klasy pozytywnej). Podziel je (`test_size=0.25`, `random_state=42`, `stratify`) i wytrenuj dwie regresje logistyczne: zwykłą oraz z `class_weight='balanced'`.

Dla obu wydrukuj na zbiorze testowym **accuracy** i **recall** klasy pozytywnej (3 miejsca). Co się dzieje z accuracy, a co z recall?

__Oczekiwany wynik:__

```
bez wag 0.946 0.0
balanced 0.673 0.721
```

__Wskazówka:__ Wagi klas zwiększają „karę” za pomyłki na klasie rzadkiej - zwykle rośnie recall, a accuracy może spaść.

In [7]:
X_imb, y_imb = make_classification(
    n_samples=5000, n_features=10, n_informative=4, weights=[0.95, 0.05],
    class_sep=0.8, random_state=42
)
print(np.bincount(y_imb))

[4729  271]


In [8]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_imb, y_imb, test_size=0.25, random_state=42, stratify=y_imb
)
for name, weight in [('bez wag', None), ('balanced', 'balanced')]:
    model = LogisticRegression(class_weight=weight, max_iter=1000).fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    print(name, round(accuracy_score(y_te, y_pred), 3), round(recall_score(y_te, y_pred), 3))

bez wag 0.946 0.0
balanced 0.673 0.721


### Ćwiczenie 46
Jeden podział na zbiór treningowy i testowy daje tylko jedną ocenę. Oceń `DecisionTreeClassifier(max_depth=3, random_state=42)` na całym zbiorze `cancer` (`X_c`, `y_c`) za pomocą **5-krotnej walidacji krzyżowej** (`StratifiedKFold` z `shuffle=True`, `random_state=42`).

Wydrukuj wyniki pięciu foldów oraz ich średnią i odchylenie standardowe (3 miejsca).

__Oczekiwany wynik:__

```
[0.921 0.886 0.947 0.93  0.938]
0.924 0.021
```

__Wskazówka:__ Funkcja `cross_val_score(model, X, y, cv=cv)` zwraca tablicę wyników dla kolejnych foldów.

In [10]:
X_c, y_c = load_breast_cancer(return_X_y=True)
X_c.shape

(569, 30)

In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42), X_c, y_c, cv=cv)
print(scores.round(3))
print(round(scores.mean(), 3), round(scores.std(), 3))

[0.921 0.886 0.947 0.93  0.938]
0.924 0.021


### Ćwiczenie 47
Skalowanie danych **przed** walidacją krzyżową powoduje wyciek danych (*data leakage*): `StandardScaler` „widzi” obserwacje, które w danym foldzie są testowe.

Zbuduj potok `StandardScaler` + `LogisticRegression(max_iter=1000)` i oceń go na `X_c`, `y_c` 5-krotną walidacją krzyżową (`StratifiedKFold`, `shuffle=True`, `random_state=42`). Wydrukuj średnią accuracy (3 miejsca).

Dlaczego skaler musi znajdować się **wewnątrz** potoku?

__Oczekiwany wynik:__

```
0.974
```

__Wskazówka:__ W potoku skaler jest dopasowywany od nowa w każdym foldzie, tylko na jego części treningowej.

In [12]:
X_c, y_c = load_breast_cancer(return_X_y=True)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [13]:
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
scores = cross_val_score(pipe, X_c, y_c, cv=cv)
print(round(scores.mean(), 3))

0.974


### Ćwiczenie 48
Dobierz hiperparametry drzewa za pomocą `GridSearchCV` (`cv=5`, `scoring='accuracy'`) na zbiorze treningowym `make_moons`.

Siatka: `max_depth` $\in \{2, 4, 6, 8, 10\}$ oraz `min_samples_leaf` $\in \{1, 5, 10, 20\}$. Wydrukuj najlepsze parametry, najlepszy wynik walidacji (3 miejsca) oraz accuracy najlepszego modelu na zbiorze testowym (3 miejsca).

__Oczekiwany wynik:__

```
{'max_depth': 6, 'min_samples_leaf': 5}
0.95
0.93
```

__Wskazówka:__ Po `fit()` najlepszy model jest dostępny jako `best_estimator_`, a `score()` korzysta z niego od razu.

In [14]:
param_grid = {
    'max_depth': [2, 4, 6, 8, 10],
    'min_samples_leaf': [1, 5, 10, 20],
}
grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring='accuracy'
)
grid.fit(X_train, y_train)
print(grid.best_params_)
print(round(grid.best_score_, 3))
print(round(grid.score(X_test, y_test), 3))

{'max_depth': 6, 'min_samples_leaf': 5}
0.95
0.93


### Ćwiczenie 49
Dla większych przestrzeni parametrów używa się `RandomizedSearchCV`, który losuje zadaną liczbę kombinacji.

Przeszukaj hiperparametry lasu losowego (`random_state=42`): `n_estimators` z `randint(50, 300)`, `max_depth` z `randint(2, 12)` i `min_samples_leaf` z `randint(1, 20)`. Użyj `n_iter=10`, `cv=3`, `random_state=42`. Wydrukuj najlepsze parametry i accuracy na zbiorze testowym (3 miejsca).

__Oczekiwany wynik:__

```
{'max_depth': 7, 'min_samples_leaf': 2, 'n_estimators': 241}
0.934
```

__Wskazówka:__ Rozkłady (np. `randint`) zamiast list wartości, a liczbę losowań określa `n_iter`.

In [15]:
param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(2, 12),
    'min_samples_leaf': randint(1, 20),
}
search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42), param_dist,
    n_iter=10, cv=3, random_state=42, n_jobs=-1
)
search.fit(X_train, y_train)
print({k: int(v) for k, v in search.best_params_.items()})
print(round(search.score(X_test, y_test), 3))

{'max_depth': 7, 'min_samples_leaf': 2, 'n_estimators': 241}
0.934


### Ćwiczenie 50
Wytrenuj `RandomForestClassifier(n_estimators=200, random_state=42)` na zbiorze `cancer` (podział z komórki poniżej). Wyznacz ważność cech dwoma sposobami:
- wbudowaną (`feature_importances_`),
- **permutacyjną** na zbiorze testowym (`permutation_importance`, `n_repeats=10`, `random_state=42`).

Dla każdej metody wydrukuj nazwy 3 najważniejszych cech. Dlaczego ważność permutacyjna jest bardziej wiarygodna?

__Oczekiwany wynik:__

```
['worst perimeter', 'worst area', 'worst concave points']
['worst texture', 'worst perimeter', 'mean radius']
```

__Wskazówka:__ Ważność permutacyjna mierzy spadek jakości po przetasowaniu cechy na danych, których model nie widział.

In [16]:
cancer = load_breast_cancer(as_frame=True)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    cancer.data, cancer.target, test_size=0.25, random_state=42, stratify=cancer.target
)

In [17]:
forest = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xc_train, yc_train)
built_in = pd.Series(forest.feature_importances_, index=Xc_train.columns)
perm = permutation_importance(forest, Xc_test, yc_test, n_repeats=10, random_state=42)
permutation = pd.Series(perm.importances_mean, index=Xc_train.columns)
print(list(built_in.nlargest(3).index))
print(list(permutation.nlargest(3).index))

['worst perimeter', 'worst area', 'worst concave points']
['worst texture', 'worst perimeter', 'mean radius']
